# Phase 4: Causal Inference

What interventions actually CAUSE retention?


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from dowhy import CausalModel
from econml.dr import DRLearner
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier


In [2]:
df = pd.read_csv('../data/telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df['Treatment'] = (df['TechSupport'] == 'Yes').astype(int)


In [3]:
confounders = ['tenure', 'MonthlyCharges', 'SeniorCitizen']
ps_model = LogisticRegression()
ps_model.fit(df[confounders], df['Treatment'])
df['Propensity_Score'] = ps_model.predict_proba(df[confounders])[:, 1]
treated = df[df['Treatment'] == 1]
control = df[df['Treatment'] == 0]
nn = NearestNeighbors(n_neighbors=1)
nn.fit(control[['Propensity_Score']])
matched_control = control.iloc[nn.kneighbors(treated[['Propensity_Score']])[1].flatten()]
ate = treated['Churn'].mean() - matched_control['Churn'].mean()
print(f'ATE (PSM): {ate:.4f}')


ATE (PSM): -0.1301


In [4]:
model = CausalModel(data=df, treatment='Treatment', outcome='Churn', common_causes=['tenure', 'MonthlyCharges', 'SeniorCitizen'])
identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
estimate = model.estimate_effect(identified_estimand, method_name='backdoor.propensity_score_matching', target_units='ate')
print(f'DoWhy Estimate: {estimate.value:.4f}')


DoWhy Estimate: -0.1041


In [5]:
X_uplift = df[confounders].values
T_uplift = df['Treatment'].values
Y_uplift = df['Churn'].values
est = DRLearner(model_regression=GradientBoostingRegressor(), model_propensity=GradientBoostingClassifier(), model_final=GradientBoostingRegressor(), cv=3)
est.fit(Y_uplift, T_uplift, X=X_uplift)
uplift_scores = est.effect(X_uplift)
df['UpliftScore'] = uplift_scores
df.to_csv('../data/processed_with_uplift.csv', index=False)
